# Eksperyment: SliceAware — pierwszy wariant slice-aware

## Cel
Model bazowy `tennis_model.py` osiąga ~64% match accuracy, ale `Experiment_ModelSlice` pokazał, że w **konkretnych podgrupach** wynik jest gorszy:
- **Best of 5** — mecze pięciosetowe (Grand Slamy)
- **Round = QF** — ćwierćfinały
- **L-vs-R** — leworęczny vs praworęczny

Na tych słabych slice'ach model traci 5-15 p.p. accuracy względem średniej. Pomysł: dodać cechy, które **konkretnie celują w te konteksty**.

## Co dodajemy (32 nowe cechy)
1. **Best of 5 form/experience** — jak gracz radzi sobie w meczach Bo5 (osobna forma, nie zlana z Bo3)
2. **Late round form/experience** — forma w fazie QF/SF/F (presja)
3. **vs opponent hand form** — jak gracz radzi sobie konkretnie z lewo/praworęcznym
4. **QF form/experience/surface form** — bardziej granularne dla ćwierćfinałów
5. **vs opp hand balance** — wins minus losses przeciwko ręczności rywala

Wszystkie liczone z **expanding window** (tylko mecze przed bieżącym), z fallbackiem na ogólną formę gdy brak historii kontekstowej.

In [1]:
import bisect
import contextlib
import io
import os
import runpy
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

LATE_ROUNDS = {"QF", "SF", "BR", "F"}

## 1. Reuse baseline pipeline

Uruchamiamy `tennis_model.py` żeby dostać:
- ten sam podział train/val/test (chronologiczny 60/20/20)
- te same `df_train_raw`/`df_val_raw`/`df_test_raw` z policzonymi już cechami dynamicznymi (`w_form`, `l_form`, `w_surface_form` itd.)
- tuned hyperparams z RandomizedSearchCV (`baseline_search.best_params_`)
- baseline metryki do porównania

`contextlib.redirect_stdout` wycisza output baseline'u — interesują nas tylko deltas.

In [2]:
BASE_SCRIPT = Path("../src/tennis_model.py").resolve()

captured = io.StringIO()
with contextlib.redirect_stdout(captured):
    namespace = runpy.run_path(str(BASE_SCRIPT))

base_cols = list(namespace["cols_base"])
symmetrize_data = namespace["symmetrize_data"]
baseline_search = namespace["search"]
baseline_val_acc = float(namespace["val_acc"])
baseline_test_acc = float(namespace["test_acc"])
baseline_match_accuracy = float(namespace["match_accuracy"])

print(f"Baseline match accuracy: {baseline_match_accuracy:.4f}")
print(f"Baseline val accuracy:   {baseline_val_acc:.4f}")
print(f"Baseline test accuracy:  {baseline_test_acc:.4f}")
print(f"Baseline best_params:    {baseline_search.best_params_}")

Baseline match accuracy: 0.6566
Baseline val accuracy:   0.6106
Baseline test accuracy:  0.6604
Baseline best_params:    {'n_estimators': 300, 'min_samples_split': 2, 'min_samples_leaf': 8, 'max_samples': 0.9, 'max_features': 'sqrt', 'max_depth': 10, 'bootstrap': True}


## 2. Player history index — dlaczego jest potrzebny

Dodajemy ~16 cech kontekstowych na każdy mecz. Każda wymaga filtrowania historii: 'mecze gracza X w Bo5', 'mecze gracza X przeciw leworęcznym', itd. Naiwnie: skan 18 000 wierszy historii × 16 wywołań × 2700 meczów = setki milionów porównań stringów.

**Rozwiązanie**: pre-built index `player_name -> sorted list of row indices`. Dla każdego meczu robimy `bisect_left` żeby dostać prefix indeksów `< cutoff` (tylko mecze rozegrane wcześniej). Lookup spada z O(N) do O(log K).

In [3]:
def build_player_index(full_sequence):
    """player_name -> posortowana lista indeksow wierszy gracza w full_sequence."""
    winners = full_sequence["winner_name"].to_numpy()
    losers = full_sequence["loser_name"].to_numpy()
    indices = defaultdict(list)
    for idx in range(len(full_sequence)):
        indices[winners[idx]].append(idx)
        if losers[idx] != winners[idx]:
            indices[losers[idx]].append(idx)
    return indices

def get_player_history_via_index(player_name, full_sequence, player_index, cutoff):
    """Zwraca DataFrame z meczami gracza < cutoff (rozegranymi wczesniej)."""
    all_indices = player_index.get(player_name, [])
    if not all_indices:
        return full_sequence.iloc[[]]
    end = bisect.bisect_left(all_indices, cutoff)
    if end == 0:
        return full_sequence.iloc[[]]
    return full_sequence.iloc[all_indices[:end]]

## 3. Funkcje kontekstowe — form, experience, balance

Trzy typy metryk:
- **context form** — win rate gracza w danym kontekście (np. tylko Bo5)
- **context experience** — ile gracz rozegrał meczów w kontekście (skalowane do [0,1])
- **context balance** — różnica wins-losses przeciwko konkretnej ręczności (znormalizowana)

`fallback` na ogólną formę gracza gdy nie ma minimum 2-3 meczów w kontekście — chroni przed `0.5` artefaktem dla nowych graczy.

In [4]:
def _apply_context_filters(player_history, player_name, *, best_of=None, rounds=None, surface=None, opponent_hand=None):
    if best_of is not None:
        player_history = player_history[player_history["best_of"] == best_of]
    if rounds is not None:
        player_history = player_history[player_history["round"].isin(rounds)]
    if surface is not None:
        player_history = player_history[player_history["surface"] == surface]
    if opponent_hand is not None:
        mask = (
            ((player_history["winner_name"] == player_name) & (player_history["loser_hand"] == opponent_hand)) |
            ((player_history["loser_name"] == player_name) & (player_history["winner_hand"] == opponent_hand))
        )
        player_history = player_history[mask]
    return player_history

def calculate_context_form(player_name, history, *, best_of=None, rounds=None, surface=None,
                            opponent_hand=None, window=12, min_matches=3, fallback=0.5):
    ph = _apply_context_filters(history, player_name, best_of=best_of, rounds=rounds,
                                  surface=surface, opponent_hand=opponent_hand).tail(window)
    if len(ph) < min_matches:
        return fallback
    return float((ph["winner_name"] == player_name).sum() / len(ph))

def calculate_context_experience(player_name, history, *, best_of=None, rounds=None,
                                   surface=None, window=30, scale=8):
    ph = _apply_context_filters(history, player_name, best_of=best_of, rounds=rounds,
                                  surface=surface, opponent_hand=None)
    n = len(ph.tail(window))
    return float(min(n / scale, 1.0))

def calculate_context_balance(player_name, history, *, opponent_hand, window=20, min_matches=3, fallback=0.0):
    ph = _apply_context_filters(history, player_name, opponent_hand=opponent_hand).tail(window)
    if len(ph) < min_matches:
        return fallback
    wins = (ph["winner_name"] == player_name).sum()
    losses = len(ph) - wins
    return float((wins - losses) / len(ph))

## 4. Liczenie cech kontekstowych — główna pętla

Dla każdego meczu w split:
1. Buduje cutoff = absolute index of current match
2. Pobiera per-player history via index (raz dla winner, raz dla loser)
3. Liczy 16 cech kontekstowych
4. Zapisuje jako `w_*`/`l_*` (perspektywa zwycięzcy/przegranego — będzie zmapowana na p1/p2 w symetryzacji)

In [5]:
def add_targeted_slice_features(df_subset, historical_data, base_cols):
    df_subset = df_subset.copy()
    full_sequence = pd.concat([historical_data[base_cols], df_subset[base_cols]], ignore_index=True)
    start_idx = len(historical_data)
    player_index = build_player_index(full_sequence)

    feature_rows = []
    for i in range(len(df_subset)):
        row = df_subset.iloc[i]
        cutoff = start_idx + i
        winner_name, loser_name, surface = row["winner_name"], row["loser_name"], row["surface"]
        w_form, l_form = float(row["w_form"]), float(row["l_form"])
        w_sf, l_sf = float(row["w_surface_form"]), float(row["l_surface_form"])

        w_hist = get_player_history_via_index(winner_name, full_sequence, player_index, cutoff)
        l_hist = get_player_history_via_index(loser_name, full_sequence, player_index, cutoff)

        feature_rows.append({
            "w_best_of5_form":   calculate_context_form(winner_name, w_hist, best_of=5, window=8, min_matches=2, fallback=w_form),
            "l_best_of5_form":   calculate_context_form(loser_name,  l_hist, best_of=5, window=8, min_matches=2, fallback=l_form),
            "w_late_round_form": calculate_context_form(winner_name, w_hist, rounds=LATE_ROUNDS, window=8, min_matches=2, fallback=w_form),
            "l_late_round_form": calculate_context_form(loser_name,  l_hist, rounds=LATE_ROUNDS, window=8, min_matches=2, fallback=l_form),
            "w_best_of5_experience":   calculate_context_experience(winner_name, w_hist, best_of=5, window=20, scale=6),
            "l_best_of5_experience":   calculate_context_experience(loser_name,  l_hist, best_of=5, window=20, scale=6),
            "w_late_round_experience": calculate_context_experience(winner_name, w_hist, rounds=LATE_ROUNDS, window=20, scale=6),
            "l_late_round_experience": calculate_context_experience(loser_name,  l_hist, rounds=LATE_ROUNDS, window=20, scale=6),
            "w_vs_opp_hand_form": calculate_context_form(winner_name, w_hist, opponent_hand=row["loser_hand"],  window=12, min_matches=3, fallback=w_form),
            "l_vs_opp_hand_form": calculate_context_form(loser_name,  l_hist, opponent_hand=row["winner_hand"], window=12, min_matches=3, fallback=l_form),
            "w_qf_form": calculate_context_form(winner_name, w_hist, rounds={"QF"}, window=6, min_matches=1, fallback=w_form),
            "l_qf_form": calculate_context_form(loser_name,  l_hist, rounds={"QF"}, window=6, min_matches=1, fallback=l_form),
            "w_qf_experience": calculate_context_experience(winner_name, w_hist, rounds={"QF"}, window=16, scale=4),
            "l_qf_experience": calculate_context_experience(loser_name,  l_hist, rounds={"QF"}, window=16, scale=4),
            "w_qf_surface_form": calculate_context_form(winner_name, w_hist, rounds={"QF"}, surface=surface, window=4, min_matches=1, fallback=w_sf),
            "l_qf_surface_form": calculate_context_form(loser_name,  l_hist, rounds={"QF"}, surface=surface, window=4, min_matches=1, fallback=l_sf),
            "w_vs_opp_hand_surface_form": calculate_context_form(winner_name, w_hist, surface=surface, opponent_hand=row["loser_hand"],  window=8, min_matches=2, fallback=w_sf),
            "l_vs_opp_hand_surface_form": calculate_context_form(loser_name,  l_hist, surface=surface, opponent_hand=row["winner_hand"], window=8, min_matches=2, fallback=l_sf),
            "w_vs_opp_hand_balance": calculate_context_balance(winner_name, w_hist, opponent_hand=row["loser_hand"],  window=20, min_matches=3),
            "l_vs_opp_hand_balance": calculate_context_balance(loser_name,  l_hist, opponent_hand=row["winner_hand"], window=20, min_matches=3),
        })

    feature_frame = pd.DataFrame(feature_rows)
    return pd.concat([df_subset.reset_index(drop=True), feature_frame], axis=1)

df_history_base = namespace["df_history_base"].copy()
df_train_raw = add_targeted_slice_features(namespace["df_train_raw"].copy(), df_history_base, base_cols)
history_val = pd.concat([df_history_base, df_train_raw[base_cols]], ignore_index=True)
df_val_raw = add_targeted_slice_features(namespace["df_val_raw"].copy(), history_val, base_cols)
history_test = pd.concat([df_history_base, df_train_raw[base_cols], df_val_raw[base_cols]], ignore_index=True)
df_test_raw = add_targeted_slice_features(namespace["df_test_raw"].copy(), history_test, base_cols)

print(f"Train: {len(df_train_raw)} meczow z dodanymi cechami kontekstowymi")
print(f"Val:   {len(df_val_raw)}")
print(f"Test:  {len(df_test_raw)}")

Train: 1588 meczow z dodanymi cechami kontekstowymi
Val:   529
Test:  530


## 5. Symetryzacja + mapping p1/p2

Baseline `symmetrize_data` zamienia każdy mecz w dwie perspektywy (p1=winner z y=1, p1=loser z y=0). My musimy zmapować nasze `w_*`/`l_*` cechy na `p1_*`/`p2_*` zgodnie z perspektywą:
- gdy `y=1` (p1 jest faktycznym zwycięzcą): `p1_best_of5_form = w_best_of5_form`
- gdy `y=0` (p1 jest faktycznym przegranym): `p1_best_of5_form = l_best_of5_form`

`np.where(mask, a, b)` robi to wektorowo. Diff = p1 - p2 oddaje sygnał kierunkowy.

In [6]:
SYMMETRIC_FEATURE_SPECS = [
    ("best_of5_form", "best_of5_form_diff"),
    ("best_of5_experience", "best_of5_experience_diff"),
    ("late_round_form", "late_round_form_diff"),
    ("late_round_experience", "late_round_experience_diff"),
    ("vs_opp_hand_form", "opp_hand_form_diff"),
    ("qf_form", "qf_form_diff"),
    ("qf_experience", "qf_experience_diff"),
    ("qf_surface_form", "qf_surface_form_diff"),
    ("vs_opp_hand_surface_form", "opp_hand_surface_form_diff"),
    ("vs_opp_hand_balance", "opp_hand_balance_diff"),
]

def attach_targeted_features(symmetrized, raw):
    helper_cols = ["match_id", "round", "winner_hand", "loser_hand"]
    for name, _ in SYMMETRIC_FEATURE_SPECS:
        helper_cols.extend([f"w_{name}", f"l_{name}"])

    enriched = symmetrized.merge(raw[helper_cols], on="match_id", how="left", validate="many_to_one")
    winner_mask = enriched["y"] == 1

    enriched["is_best_of5"] = (enriched["best_of"] == 5).astype(int)
    enriched["is_qf"] = (enriched["round"] == "QF").astype(int)
    enriched["is_lefty_matchup"] = (enriched["winner_hand"] != enriched["loser_hand"]).astype(int)

    for name, diff_col in SYMMETRIC_FEATURE_SPECS:
        p1, p2 = f"p1_{name}", f"p2_{name}"
        enriched[p1] = np.where(winner_mask, enriched[f"w_{name}"], enriched[f"l_{name}"])
        enriched[p2] = np.where(winner_mask, enriched[f"l_{name}"], enriched[f"w_{name}"])
        enriched[diff_col] = enriched[p1] - enriched[p2]

    drop = ["round", "winner_hand", "loser_hand"]
    for name, _ in SYMMETRIC_FEATURE_SPECS:
        drop.extend([f"w_{name}", f"l_{name}"])
    return enriched.drop(columns=drop)

train_data = attach_targeted_features(symmetrize_data(df_train_raw, shuffle=True), df_train_raw)
val_data = attach_targeted_features(symmetrize_data(df_val_raw, shuffle=True), df_val_raw)
test_data = attach_targeted_features(symmetrize_data(df_test_raw, shuffle=True), df_test_raw)

TARGETED_FEATURES = ["is_best_of5", "is_qf", "is_lefty_matchup"]
for name, diff in SYMMETRIC_FEATURE_SPECS:
    TARGETED_FEATURES.extend([f"p1_{name}", f"p2_{name}", diff])

features = list(namespace["features"]) + TARGETED_FEATURES
print(f"Liczba cech: {len(features)} (baseline: {len(namespace['features'])}, dodane: {len(TARGETED_FEATURES)})")

Liczba cech: 73 (baseline: 40, dodane: 33)


## 6. Trening RF z baseline hyperparams

Kluczowy detal: **NIE robimy nowego RandomizedSearchCV**. Bierzemy `baseline_search.best_params_` i trenujemy z nimi nowy RF na rozszerzonym zbiorze cech. To zapewnia, że delta accuracy wynika **z cech, nie z hyperparams** — porównanie jest uczciwe.

In [7]:
X_train, y_train = train_data[features], train_data["y"]
X_val, y_val = val_data[features], val_data["y"]
X_test, y_test = test_data[features], test_data["y"]

best_rf = RandomForestClassifier(**baseline_search.best_params_, n_jobs=-1, random_state=namespace["RANDOM_STATE"])
best_rf.fit(X_train, y_train)

val_pred = best_rf.predict(X_val)
test_pred = best_rf.predict(X_test)
test_proba = best_rf.predict_proba(X_test)

val_acc = accuracy_score(y_val, val_pred)
test_acc = accuracy_score(y_test, test_pred)
print(f"Val accuracy:  {val_acc:.4f}")
print(f"Test accuracy: {test_acc:.4f}")

Val accuracy:  0.6248
Test accuracy: 0.6453


## 7. Match-level accuracy + porównanie z baseline

Match-level: metryka symetryczna z baseline'u (`compute_symmetric_match_evaluation`) — łączy obie perspektywy meczu po `match_id` i uśrednia prawdopodobieństwo rzeczywistego zwycięzcy. Deterministyczna (nie zależy od shuffle) i odporna na arbitralny wybór p1/p2; stara wersja jednostronna (tylko `y=1`) zawyżała wynik.

In [8]:
test_data["p1_win_probability"] = test_proba[:, 1]
# Metryka symetryczna (Sprint 1): obie perspektywy meczu po match_id,
# usredniona proba zwyciezcy -- spojnie z baseline
# (stara wersja jednostronna, tylko y==1, zawyzala wynik).
compute_symmetric_match_evaluation = namespace["compute_symmetric_match_evaluation"]
winner_perspective, match_accuracy = compute_symmetric_match_evaluation(test_data)

print("=== POROWNANIE Z BASELINE ===")
print(f"Validation:  baseline={baseline_val_acc:.4f}  | sliceaware={val_acc:.4f}  | delta={val_acc - baseline_val_acc:+.4f}")
print(f"Test:        baseline={baseline_test_acc:.4f}  | sliceaware={test_acc:.4f}  | delta={test_acc - baseline_test_acc:+.4f}")
print(f"Match-level: baseline={baseline_match_accuracy:.4f}  | sliceaware={match_accuracy:.4f}  | delta={match_accuracy - baseline_match_accuracy:+.4f}")

=== POROWNANIE Z BASELINE ===
Validation:  baseline=0.6106  | sliceaware=0.6248  | delta=+0.0142
Test:        baseline=0.6604  | sliceaware=0.6453  | delta=-0.0151
Match-level: baseline=0.6566  | sliceaware=0.6528  | delta=-0.0038


## 8. Ważność nowych cech

Sprawdzamy, które z dodanych cech faktycznie weszły do drzew RF. Jeśli wszystkie mają importance ~0, to znaczy że są redundantne wobec baseline'u i nasza intuicja była zła. Jeśli kilka ma wysoką importance — model uznał je za użyteczne.

In [9]:
fi = pd.DataFrame({"feature": features, "importance": best_rf.feature_importances_})
new_fi = fi[fi["feature"].isin(TARGETED_FEATURES)].sort_values("importance", ascending=False)
print("=== WAZNOSC NOWYCH CECH (top 15) ===")
print(new_fi.head(15).to_string(index=False))

=== WAZNOSC NOWYCH CECH (top 15) ===
                    feature  importance
      opp_hand_balance_diff    0.027082
         best_of5_form_diff    0.025552
         opp_hand_form_diff    0.025514
 opp_hand_surface_form_diff    0.015363
     p2_vs_opp_hand_balance    0.013721
               qf_form_diff    0.011279
     p1_vs_opp_hand_balance    0.010419
       late_round_form_diff    0.010384
           p2_best_of5_form    0.009594
p1_vs_opp_hand_surface_form    0.009367
           p1_best_of5_form    0.009301
        p1_vs_opp_hand_form    0.008843
       qf_surface_form_diff    0.008654
p2_vs_opp_hand_surface_form    0.008006
        p2_vs_opp_hand_form    0.007965


## 9. Wnioski

**Wynik bieżący (RANDOM_STATE=42, sezon 2025, metryka symetryczna)**:
- Baseline match accuracy: **65.66%**
- SliceAware match accuracy: **65.28%**
- **Delta: -0.38 p.p.** -- praktycznie bez zmian, lekkie pogorszenie

Charakterystyczny rozjazd: walidacja +1.42 p.p., ale test -1.51 p.p. (accuracy binarna) -- sygnał, że 33 dodatkowe cechy dopasowują się do walidacji, a nie generalizują.

**Dlaczego delta jest blisko zera mimo dodania 33 cech kontekstowych:**

- Cechy kontekstowe mają silny `fallback` na ogólną formę -- dla graczy bez historii w danym kontekście redukują się do informacji, którą baseline już ma.
- Random Forest uśrednia: 33 cechy, z których żadna nie dominuje (najwyższa ważność nowej cechy: `opp_hand_balance_diff` 0.027), rozcieńczają sygnał.
- Wnioski per slice z historycznego runa 2024 (np. +8.6 p.p. na L-vs-R × 500) nie przenoszą się między sezonami -- ModelSlice na 2025 pokazuje inne słabe miejsca niż na 2024.

**Wniosek metodologiczny**: shotgun approach (szeroki zestaw cech pod 3 slice'y naraz) nie poprawia modelu. Na tym sezonie najlepiej wypada focused wariant BestOf5 v1 (+0.94 p.p., patrz `TPM_Experiment_SliceAware_BestOf5_v1`), ale walk-forward 2020-2025 nie potwierdza istotności żadnego wariantu (sliceaware: -0.26 p.p., p=0.61).

> Historyczne wyniki z sezonu 2024 ze starą, jednostronną metryką (baseline 61.02%, sliceaware 60.85%) są nieporównywalne z powyższymi -- szczegóły w docs/WYNIKI_SPRINTOW.md.